In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
df_if = pd.read_parquet(Path(r"D:\ProyectoAnalisisElectrico\BarrasEstaciones\infraestructura_localizada.parquet"))
df_cmg = pd.read_parquet(Path(r"D:\ProyectoAnalisisElectrico\Cmg\Cmg_period.parquet"))

In [3]:
df_if.head(3)

,ID_SE,Nombre_SE,Nombre Coordinado_SE,Nemotecnico_SE,Región,Provincia,Comuna,Nombre_BA,Nemotecnico_BA,Macrozona,nombre_barra,nombre_barra_cmg,tension,BARRA_INFOTECNICA
0,499.0,S/E ALHUE,CGE TRANSMISIÓN S.A.,SE209T0058,Metropolitana de Santiago,Melipilla,Alhué,BA S/E ALHUE 66KV,BA01T0058SE209T0058,Centro,ALHUE,ALHUE_________066,66,BA S/E ALHUE 66KV
1,356.0,S/E EL PEÑON,CGE TRANSMISIÓN S.A.,SE005T0058,Coquimbo,Elqui,Coquimbo,BA S/E EL PEÑON 66KV,BA07T0058SE005T0058,Norte Chico,E.PENON,E.PENON_______066,66,BA S/E EL PEÑON 66KV
3,377.0,S/E ANDACOLLO,CGE TRANSMISIÓN S.A.,SE026T0058,Coquimbo,Elqui,Andacollo,BA S/E ANDACOLLO 66KV BP1,BA03T0058SE026T0058,Norte Chico,ANDACOLLO,ANDACOLLO_____066,66,BA S/E ANDACOLLO 66KV BP1


In [4]:
df_cmg.head()

,nombre_barra,tension,nombre_barra_cmg,HORA,CMg[CLP/KWh],CMg[USD/MWh],Año,Mes,Periodo,CMg[CLP/KWh]_mes,CMg[USD/MWh]_mes,CMg[CLP/KWh]_period,CMg[USD/MWh]_period
0,A,100,A_____________100,1,71.568910,75.925060,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
1,A,100,A_____________100,2,69.478102,73.703558,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
2,A,100,A_____________100,3,77.806368,82.567770,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
3,A,100,A_____________100,4,79.085300,83.945624,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068
4,A,100,A_____________100,5,72.378804,76.801453,2025,05,2505_2604,53.099218,56.345685,38.901679,41.912068


In [5]:
df = pd.merge(left = df_cmg, right = df_if[["Macrozona", "nombre_barra_cmg"]], on="nombre_barra_cmg", how="left")


In [6]:
df_loc = df[~df["Macrozona"].isna()].copy()
df_nloc = df[df["Macrozona"].isna()].copy()
len(df_loc), len(df_nloc)

(318938, 89546)

In [7]:
df_barras_unicas = df_loc[['nombre_barra_cmg', 'Macrozona']].drop_duplicates()
barras_train, barras_test = train_test_split(
    df_barras_unicas['nombre_barra_cmg'], 
    test_size=0.20,         # 20% para el set de prueba
    random_state=42,        # Semilla para que el resultado sea repetible
    stratify=df_barras_unicas['Macrozona'] # Asegura equidad regional
)

df_train = df_loc[df_loc['nombre_barra_cmg'].isin(barras_train)].copy()
df_test = df_loc[df_loc['nombre_barra_cmg'].isin(barras_test)].copy()

In [8]:
df_train["CMg[USD/MWh]_period_macrozona"] = df_train.groupby(by=["Macrozona", "Periodo"])["CMg[USD/MWh]"].transform("mean")

In [9]:
df_ref = df_train[["nombre_barra_cmg", "CMg[USD/MWh]_period", "CMg[USD/MWh]_period_macrozona", "Macrozona"]].drop_duplicates()

In [10]:
df_ref["Dist"] = np.abs(df_ref["CMg[USD/MWh]_period"] - df_ref["CMg[USD/MWh]_period_macrozona"])

In [11]:
df_ref.head()

,nombre_barra_cmg,CMg[USD/MWh]_period,CMg[USD/MWh]_period_macrozona,Macrozona,Dist
0,A_____________100,41.912068,42.391998,Norte Grande,0.479931
24,A.BLANCAS_____013,53.611078,53.792578,Centro Sur,0.181499
48,A.BLANCAS_____066,53.360695,53.792578,Centro Sur,0.431883
96,A.DERIBERA____066,52.875406,53.792578,Centro Sur,0.917172
120,A.DERIBERA____154,52.379205,53.792578,Centro Sur,1.413373


In [12]:
indices_minimos = df_ref.groupby("Macrozona")["Dist"].idxmin()
df_reps = df_ref.loc[indices_minimos]
df_reps
df_reps.to_parquet(Path(r"D:\ProyectoAnalisisElectrico\Cmg\referencias_macrozona.parquet"), engine = "pyarrow")

In [13]:
df_t = df_test[['nombre_barra_cmg', 'Macrozona', 'CMg[USD/MWh]_period']].drop_duplicates().copy()
refs = df_reps.set_index('Macrozona')['CMg[USD/MWh]_period'].to_dict()

def clasificar_porcentaje(val, ref_dict):
    dists = {z: abs(val - v) / (v + 0.001) for z, v in ref_dict.items()}
    return min(dists, key=dists.get)

df_t['Pred'] = df_t['CMg[USD/MWh]_period'].apply(lambda x: clasificar_porcentaje(x, refs))

ok = (df_t['Macrozona'] == df_t['Pred']).sum()
tot = len(df_t)
acc = (ok / tot) * 100

print(f"Total: {tot} | Aciertos: {ok} | Accuracy: {acc:.2f}%")

Total: 222 | Aciertos: 137 | Accuracy: 61.71%


In [14]:
matriz = pd.crosstab(
    df_t['Macrozona'], 
    df_t['Pred'], 
    rownames=['Zona Real'], 
    colnames=['Lo que dijo el Modelo']
)

print(matriz)
print("\n" + "="*40 + "\n")

df_errores = df_t[df_t['Macrozona'] != df_t['Pred']]
top_errores = df_errores.groupby(['Macrozona', 'Pred']).size().reset_index(name='Cantidad de Fallos')
top_errores = top_errores.sort_values(by='Cantidad de Fallos', ascending=False).reset_index(drop=True)

print(top_errores.head(10))

Lo que dijo el Modelo  Centro  Centro Sur  Norte Chico  Norte Grande  Sur
Zona Real                                                                
Centro                     45          17            1             0    0
Centro Sur                 20          39            0             0    7
Norte Chico                 0           0            9            16    0
Norte Grande                0           0           11            23    0
Sur                         9           2            2             0   21


      Macrozona          Pred  Cantidad de Fallos
0    Centro Sur        Centro                  20
1        Centro    Centro Sur                  17
2   Norte Chico  Norte Grande                  16
3  Norte Grande   Norte Chico                  11
4           Sur        Centro                   9
5    Centro Sur           Sur                   7
6           Sur    Centro Sur                   2
7           Sur   Norte Chico                   2
8        Centro   Norte Chico 